# Test Set Analysis

This notebook performs analysis on the test set:
1. Per-Class Performance Metrics (Dice, IoU, F1)
2. Best/Median/Worst Sample Identification  
3. Visual Results for All Test Samples

## 1. Imports and Setup

In [66]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import directed_hausdorff
from scipy.ndimage import distance_transform_edt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set publication-quality defaults
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0

print("Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Imports successful
PyTorch version: 2.5.1+cu121
CUDA available: True


## 2. Configuration

In [67]:
class Config:
    """Configuration for analysis"""
    
    # Paths - UPDATED TO USE CORRECT MODEL FROM TRAINING
    MODEL_PATH = Path("../../models/best_model.pth")
    DATA_SPLIT_PATH = Path("../../models/data_split.json")
    OUTPUT_DIR = Path("../../results/analysis_results")
    
    # Model settings
    ENCODER_NAME = 'resnet34'
    INPUT_SIZE = (512, 512)
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Class names
    CLASS_NAMES = ['Background', 'Tissue', 'OS', 'Vaginal']
    CLASS_COLORS = {
        0: [0, 0, 0],        # Background - black
        1: [0, 0, 255],      # Tissue - blue
        2: [0, 255, 0],      # OS - green
        3: [255, 0, 0]       # Vaginal - red
    }
    
    # Analysis settings
    HAUSDORFF_PERCENTILE = 95
    CONFIDENCE_LEVEL = 0.95

# Create output directory
Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Configuration set")
print(f"  Device: {Config.DEVICE}")
print(f"  Model: {Config.MODEL_PATH}")
print(f"  Data split: {Config.DATA_SPLIT_PATH}")
print(f"  Output directory: {Config.OUTPUT_DIR}")

✓ Configuration set
  Device: cuda
  Model: ..\..\models\best_model.pth
  Data split: ..\..\models\data_split.json
  Output directory: ..\..\results\analysis_results


## 3. Model Architecture (Same as Training)

In [68]:
class UNet(nn.Module):
    """Simple UNet - matches training architecture exactly"""

    def __init__(self, num_classes):
        super().__init__()
        # Using simple ResNet34 encoder
        self.encoder = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        self.enc_layers = list(self.encoder.children())

        self.enc1 = nn.Sequential(*self.enc_layers[:3])  # 64
        self.enc2 = nn.Sequential(*self.enc_layers[3:5])  # 64
        self.enc3 = self.enc_layers[5]  # 128
        self.enc4 = self.enc_layers[6]  # 256
        self.enc5 = self.enc_layers[7]  # 512

        self.dec4 = self._block(512 + 256, 256)
        self.dec3 = self._block(256 + 128, 128)
        self.dec2 = self._block(128 + 64, 64)
        self.dec1 = self._block(64 + 64, 32)

        self.final = nn.Conv2d(32, num_classes, 1)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        d4 = F.interpolate(e5, size=e4.shape[2:], mode='bilinear')
        d4 = self.dec4(torch.cat([d4, e4], 1))

        d3 = F.interpolate(d4, size=e3.shape[2:], mode='bilinear')
        d3 = self.dec3(torch.cat([d3, e3], 1))

        d2 = F.interpolate(d3, size=e2.shape[2:], mode='bilinear')
        d2 = self.dec2(torch.cat([d2, e2], 1))

        d1 = F.interpolate(d2, size=e1.shape[2:], mode='bilinear')
        d1 = self.dec1(torch.cat([d1, e1], 1))

        out = F.interpolate(d1, scale_factor=2, mode='bilinear')
        return self.final(out)

print("✓ Model architecture defined (matches training)")

✓ Model architecture defined (matches training)


## 4. Data Loading Functions

In [69]:
def extract_m11_and_mask(npz_path: Path) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """Extract M11 and 4-class mask from NPZ file."""
    try:
        with np.load(npz_path, allow_pickle=True) as data:
            # Extract M11
            m11 = None
            if 'nM11s' in data:
                m11 = np.array(data['nM11s'])
            elif 'M11s' in data:
                m11_raw = np.array(data['M11s'])
                m11_min, m11_max = m11_raw.min(), m11_raw.max()
                if m11_max > m11_min:
                    m11 = (m11_raw - m11_min) / (m11_max - m11_min)
                else:
                    m11 = np.zeros_like(m11_raw, dtype=np.float32)
            elif 'nM' in data:
                nM = np.array(data['nM'])
                if nM.ndim == 4 and nM.shape[-2:] == (4, 4):
                    m11 = nM[:, :, 0, 0]
                elif nM.ndim == 3 and nM.shape[-1] == 16:
                    m11 = nM[:, :, 0]

            if m11 is None or m11.ndim != 2:
                return None

            # Handle NaN and Inf values - replace with 0
            m11 = np.nan_to_num(m11, nan=0.0, posinf=0.0, neginf=0.0)

            # Extract masks
            tissue_mask = None
            for key in ['tissue_mask', 'annotation_mask']:
                if key in data:
                    mask_data = data[key]
                    if isinstance(mask_data, np.ndarray) and mask_data.size > 0:
                        tissue_mask = np.array(mask_data) > 0
                        break

            if tissue_mask is None:
                return None

            os_mask = None
            if 'os_mask' in data:
                mask_data = data['os_mask']
                if isinstance(mask_data, np.ndarray) and mask_data.size > 0:
                    os_mask = np.array(mask_data) > 0

            # Vaginal mask - check multiple possible keys
            vaginal_mask = None
            for key in ['vaginal_mask', 'vaginal_wall', 'vaginal_wall_mask']:
                if key in data:
                    mask_data = data[key]
                    if isinstance(mask_data, np.ndarray) and mask_data.size > 0:
                        vaginal_mask = np.array(mask_data) > 0
                        break

            # Combine into 4-class mask
            combined_mask = np.zeros_like(tissue_mask, dtype=np.int64)
            combined_mask[tissue_mask] = 1
            if os_mask is not None:
                combined_mask[os_mask] = 2
            if vaginal_mask is not None:
                combined_mask[vaginal_mask] = 3

            return m11.astype(np.float32), combined_mask.astype(np.int64)

    except Exception as e:
        print(f"Error loading {npz_path}: {e}")
        return None


def preprocess_m11(m11: np.ndarray, target_size: Tuple[int, int]) -> torch.Tensor:
    """
    Preprocess M11 for model input using ImageNet normalization.
    Matches training preprocessing exactly:
    1. Per-image min-max normalization to [0,1] (already done in extract_m11_and_mask)
    2. Resize to target_size (bilinear)
    3. Replicate to 3 channels
    4. ImageNet mean/std normalization
    """
    # Resize if needed
    if m11.shape != target_size:
        m11_torch = torch.from_numpy(m11).float().unsqueeze(0).unsqueeze(0)
        m11_torch = F.interpolate(m11_torch, size=target_size, mode='bilinear', align_corners=True)
        m11 = m11_torch.squeeze().numpy()

    # Convert to 3-channel RGB (replicate grayscale)
    m11_rgb = np.stack([m11, m11, m11], axis=0)  # [3, H, W]

    # ImageNet mean/std normalization
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
    m11_rgb = (m11_rgb - mean) / std

    tensor = torch.from_numpy(m11_rgb).float().unsqueeze(0)

    return tensor

print("✓ Data loading functions defined (using ImageNet normalization)")

✓ Data loading functions defined (using ImageNet normalization)


## 5. Metrics Calculation Functions

In [70]:
def calculate_dice_coefficient(pred: np.ndarray, gt: np.ndarray, class_id: int) -> float:
    """Calculate Dice coefficient for a specific class."""
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)

    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = pred_mask.sum() + gt_mask.sum()

    if union == 0:
        return 1.0 if intersection == 0 else 0.0

    dice = (2.0 * intersection) / union
    return dice


def calculate_iou(pred: np.ndarray, gt: np.ndarray, class_id: int) -> float:
    """Calculate Intersection over Union for a specific class."""
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)

    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()

    if union == 0:
        return 1.0 if intersection == 0 else 0.0

    iou = intersection / union
    return iou


def calculate_f1_score(pred: np.ndarray, gt: np.ndarray, class_id: int) -> float:
    """Calculate F1 score for a specific class."""
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)

    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return f1


def calculate_all_metrics(pred: np.ndarray, gt: np.ndarray, num_classes: int) -> Dict:
    """Calculate metrics for all classes."""
    metrics = {
        'overall': {},
        'per_class': {}
    }

    # Per-class metrics
    for class_id in range(num_classes):
        class_metrics = {}
        class_metrics['dice'] = calculate_dice_coefficient(pred, gt, class_id)
        class_metrics['iou'] = calculate_iou(pred, gt, class_id)
        class_metrics['f1_score'] = calculate_f1_score(pred, gt, class_id)
        metrics['per_class'][class_id] = class_metrics

    # Overall metrics

    # 1. Pixel Accuracy
    correct_pixels = np.sum(pred == gt)
    total_pixels = pred.size
    metrics['overall']['pixel_accuracy'] = correct_pixels / total_pixels

    # 2. Overall Tissue DSC (mean of non-background classes)
    tissue_dice_scores = [metrics['per_class'][i]['dice'] for i in range(1, num_classes)]
    metrics['overall']['mean_tissue_dice'] = np.mean(tissue_dice_scores)

    return metrics

print("✓ Metrics calculation functions defined")

✓ Metrics calculation functions defined


## 6. Load Model

In [71]:
# Load model
checkpoint = torch.load(Config.MODEL_PATH, map_location='cpu', weights_only=False)
num_classes = checkpoint['model_state_dict']['final.weight'].shape[0]

model = UNet(num_classes=num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(Config.DEVICE)
model.eval()

print(f"✓ Model loaded from: {Config.MODEL_PATH}")
print(f"  Number of classes: {num_classes}")
print(f"  Training epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"  Training Val Loss: {checkpoint.get('val_loss', 'N/A'):.4f}" if 'val_loss' in checkpoint else "  Training Val Loss: N/A")
print(f"  Training Val Acc: {checkpoint.get('val_acc', 'N/A'):.4f}" if 'val_acc' in checkpoint else "  Training Val Acc: N/A")

✓ Model loaded from: ..\..\models\best_model.pth
  Number of classes: 4
  Training epoch: 49
  Training Val Loss: 0.3740
  Training Val Acc: 0.9172


## 7. Run Inference and Calculate Metrics

In [72]:
@torch.no_grad()
def run_inference(npz_path: Path, model: nn.Module, device: torch.device, input_size: Tuple[int, int]) -> Dict:
    """Run inference on a single sample and calculate all metrics."""

    # Load data
    result = extract_m11_and_mask(npz_path)
    if result is None:
        return None

    m11_original, gt_mask = result
    original_shape = m11_original.shape

    # Preprocess
    input_tensor = preprocess_m11(m11_original, input_size)
    input_tensor = input_tensor.to(device)

    # Inference
    logits = model(input_tensor)
    pred_classes = torch.argmax(logits, dim=1)

    # Resize back to original size
    pred_classes = F.interpolate(
        pred_classes.unsqueeze(1).float(),
        size=original_shape,
        mode='nearest'
    ).squeeze().long()

    pred_mask = pred_classes.cpu().numpy()

    # Calculate metrics
    metrics = calculate_all_metrics(pred_mask, gt_mask, num_classes)

    return {
        'm11': m11_original,
        'ground_truth': gt_mask,
        'prediction': pred_mask,
        'metrics': metrics
    }

print(" Inference function defined")

 Inference function defined


## 8. Load Data Split and Run Analysis on Test Set

In [73]:
# Load data split
with open(Config.DATA_SPLIT_PATH, 'r') as f:
    data_split = json.load(f)

print("=" * 80)
print("DATA SPLIT INFORMATION")
print("=" * 80)
print(f"Train samples: {len(data_split['train'])}")
print(f"Val samples: {len(data_split['val'])}")
print(f"Test samples: {len(data_split['test'])}")
print("=" * 80)

# Run inference on test samples
print("\nRunning inference on TEST samples...")
test_results = []

for sample_info in tqdm(data_split['test'], desc="Test samples"):
    npz_path = Path(sample_info['path'])
    result = run_inference(npz_path, model, Config.DEVICE, Config.INPUT_SIZE)

    if result is not None:
        result['sample_name'] = sample_info['name']
        result['sample_path'] = str(npz_path)
        test_results.append(result)
    else:
        print(f"Failed to process: {sample_info['name']}")

print(f" Processed {len(test_results)} test samples")
print("\n" + "=" * 80)

DATA SPLIT INFORMATION
Train samples: 51
Val samples: 11
Test samples: 12

Running inference on TEST samples...


Test samples: 100%|██████████| 12/12 [00:02<00:00,  4.20it/s]

 Processed 12 test samples



## 9. Per-Class Performance Metrics

In [74]:
# Calculate OVERALL statistics
all_pixel_acc = [r['metrics']['overall']['pixel_accuracy'] for r in test_results]
all_tissue_dice = [r['metrics']['overall']['mean_tissue_dice'] for r in test_results]

overall_data = [
    {
        'Class': 'Overall Pixel Accuracy',
        'Dice': '-',
        'IoU': f"{np.mean(all_pixel_acc):.4f} ± {np.std(all_pixel_acc):.4f}", # Using IoU column for PA
        'F1': '-'
    },
    {
        'Class': 'Overall Tissue DSC',
        'Dice': f"{np.mean(all_tissue_dice):.4f} ± {np.std(all_tissue_dice):.4f}",
        'IoU': '-',
        'F1': '-'
    }
]

# Calculate PER-CLASS statistics
per_class_data = []

for class_id, class_name in enumerate(Config.CLASS_NAMES):
    dice_values = [r['metrics']['per_class'][class_id]['dice'] for r in test_results]
    iou_values = [r['metrics']['per_class'][class_id]['iou'] for r in test_results]
    f1_values = [r['metrics']['per_class'][class_id]['f1_score'] for r in test_results]

    row_data = {
        'Class': class_name,
        'Dice': f"{np.mean(dice_values):.4f} ± {np.std(dice_values):.4f}",
        'IoU': f"{np.mean(iou_values):.4f} ± {np.std(iou_values):.4f}",
        'F1': f"{np.mean(f1_values):.4f} ± {np.std(f1_values):.4f}"
    }

    per_class_data.append(row_data)

# Combine and create DataFrame
table_data = overall_data + per_class_data
table_df = pd.DataFrame(table_data)


print("\n" + "=" * 80)
print("PERFORMANCE (OVERALL & PER-CLASS) ON TEST SET")
print("=" * 80)
print(table_df.to_string(index=False))
print("=" * 80)

# Save table
table_df.to_csv(Config.OUTPUT_DIR / 'full_performance.csv', index=False)
print(f"\n✓ Table saved to: {Config.OUTPUT_DIR / 'full_performance.csv'}")


PERFORMANCE (OVERALL & PER-CLASS) ON TEST SET
                 Class            Dice             IoU              F1
Overall Pixel Accuracy               - 0.8971 ± 0.0551               -
    Overall Tissue DSC 0.8096 ± 0.1337               -               -
            Background 0.9237 ± 0.0942 0.8707 ± 0.1417 0.9237 ± 0.0942
                Tissue 0.8863 ± 0.0444 0.7986 ± 0.0709 0.8863 ± 0.0444
                    OS 0.8485 ± 0.0971 0.7479 ± 0.1313 0.8485 ± 0.0971
               Vaginal 0.6941 ± 0.3261 0.6067 ± 0.3054 0.6108 ± 0.3630

✓ Table saved to: ..\..\results\analysis_results\full_performance.csv


In [75]:
## 10. Best/Median/Worst Sample Identification

In [76]:
# Identify best, median, and worst samples based on overall DSC
test_results_sorted = sorted(test_results, 
                             key=lambda x: x['metrics']['overall']['mean_tissue_dice'])

# Get top 3 best, middle 3, and bottom 3 worst
n_samples = min(3, len(test_results_sorted))  # In case we have fewer than 3 samples
worst_samples = test_results_sorted[:n_samples]
median_idx = len(test_results_sorted) // 2
median_samples = test_results_sorted[median_idx-1:median_idx+2] if len(test_results_sorted) >= 3 else [test_results_sorted[median_idx]]
best_samples = test_results_sorted[-n_samples:][::-1]  # Reverse to show best first

print("\n" + "=" * 80)
print("BEST/MEDIAN/WORST SAMPLES")
print("=" * 80)

print(f"\n{'='*80}")
print("TOP {0} BEST SAMPLES".format(len(best_samples)))
print("=" * 80)
for i, sample in enumerate(best_samples, 1):
    print(f"\n#{i} - {sample['sample_name']}")
    print(f"  Overall DSC: {sample['metrics']['overall']['mean_tissue_dice']:.4f}")
    for class_id, class_name in enumerate(Config.CLASS_NAMES[1:], start=1):
        dice = sample['metrics']['per_class'][class_id]['dice']
        iou = sample['metrics']['per_class'][class_id]['iou']
        f1 = sample['metrics']['per_class'][class_id]['f1_score']
        print(f"  {class_name}: DSC={dice:.4f}, IoU={iou:.4f}, F1={f1:.4f}")

print(f"\n{'='*80}")
print("MIDDLE {0} SAMPLES".format(len(median_samples)))
print("=" * 80)
for i, sample in enumerate(median_samples, 1):
    print(f"\n#{i} - {sample['sample_name']}")
    print(f"  Overall DSC: {sample['metrics']['overall']['mean_tissue_dice']:.4f}")
    for class_id, class_name in enumerate(Config.CLASS_NAMES[1:], start=1):
        dice = sample['metrics']['per_class'][class_id]['dice']
        iou = sample['metrics']['per_class'][class_id]['iou']
        f1 = sample['metrics']['per_class'][class_id]['f1_score']
        print(f"  {class_name}: DSC={dice:.4f}, IoU={iou:.4f}, F1={f1:.4f}")

print(f"\n{'='*80}")
print("BOTTOM {0} WORST SAMPLES".format(len(worst_samples)))
print("=" * 80)
for i, sample in enumerate(worst_samples, 1):
    print(f"\n#{i} - {sample['sample_name']}")
    print(f"  Overall DSC: {sample['metrics']['overall']['mean_tissue_dice']:.4f}")
    for class_id, class_name in enumerate(Config.CLASS_NAMES[1:], start=1):
        dice = sample['metrics']['per_class'][class_id]['dice']
        iou = sample['metrics']['per_class'][class_id]['iou']
        f1 = sample['metrics']['per_class'][class_id]['f1_score']
        print(f"  {class_name}: DSC={dice:.4f}, IoU={iou:.4f}, F1={f1:.4f}")

print("\n" + "=" * 80)


BEST/MEDIAN/WORST SAMPLES

TOP 3 BEST SAMPLES

#1 - Day6_mm_results_Day6D_8B_S3
  Overall DSC: 0.9237
  Tissue: DSC=0.9457, IoU=0.8969, F1=0.9457
  OS: DSC=0.8255, IoU=0.7028, F1=0.8255
  Vaginal: DSC=1.0000, IoU=1.0000, F1=0.0000

#2 - Day15_mm_results_D15F_S6B_2
  Overall DSC: 0.9100
  Tissue: DSC=0.9084, IoU=0.8322, F1=0.9084
  OS: DSC=0.9283, IoU=0.8662, F1=0.9283
  Vaginal: DSC=0.8932, IoU=0.8070, F1=0.8932

#3 - Day15_mm_results_D15F_S6B_5
  Overall DSC: 0.9053
  Tissue: DSC=0.9024, IoU=0.8222, F1=0.9024
  OS: DSC=0.9192, IoU=0.8504, F1=0.9192
  Vaginal: DSC=0.8941, IoU=0.8086, F1=0.8941

MIDDLE 3 SAMPLES

#1 - Day18_mm_results_D18E_S2A_1
  Overall DSC: 0.8734
  Tissue: DSC=0.8599, IoU=0.7543, F1=0.8599
  OS: DSC=0.9253, IoU=0.8609, F1=0.9253
  Vaginal: DSC=0.8351, IoU=0.7169, F1=0.8351

#2 - Day0_mm_results_Day0H_2B_S7
  Overall DSC: 0.8757
  Tissue: DSC=0.8460, IoU=0.7331, F1=0.8460
  OS: DSC=0.9134, IoU=0.8407, F1=0.9134
  Vaginal: DSC=0.8676, IoU=0.7662, F1=0.8676

#3 - Day0

## 11. Visualizations for All Test Samples

In [77]:
# Generate visualizations for ALL test samples
from matplotlib.backends.backend_pdf import PdfPages

print("\nGenerating visualizations for all test samples...")

# Create PDF file for all visualizations
pdf_path = Config.OUTPUT_DIR / 'all_test_samples.pdf'

with PdfPages(pdf_path) as pdf:
    for idx, result in enumerate(tqdm(test_results, desc="Creating visualizations")):
        m11 = result['m11']
        gt_mask = result['ground_truth']
        pred_mask = result['prediction']
        sample_name = result['sample_name']
        
        # Create figure with 3 columns: M11, Ground Truth, Prediction
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # 1. M11 Image
        axes[0].imshow(m11, cmap='gray', vmin=0, vmax=1)
        axes[0].set_title('(A) M11 Image', fontweight='bold', fontsize=12)
        axes[0].axis('off')
        
        # 2. Ground Truth Mask (color-coded)
        gt_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
        for class_id, color in Config.CLASS_COLORS.items():
            gt_rgb[gt_mask == class_id] = color
        axes[1].imshow(gt_rgb)
        axes[1].set_title('(B) Ground Truth', fontweight='bold', fontsize=12)
        axes[1].axis('off')
        
        # 3. Prediction Mask (color-coded)
        pred_rgb = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
        for class_id, color in Config.CLASS_COLORS.items():
            pred_rgb[pred_mask == class_id] = color
        axes[2].imshow(pred_rgb)
        axes[2].set_title('(C) Prediction', fontweight='bold', fontsize=12)
        axes[2].axis('off')
        
        # Add title with metrics
        dsc = result['metrics']['overall']['mean_tissue_dice']
        tissue_dsc = result['metrics']['per_class'][1]['dice']
        os_dsc = result['metrics']['per_class'][2]['dice']
        vaginal_dsc = result['metrics']['per_class'][3]['dice']
        
        fig.suptitle(f"Sample {idx+1}/{len(test_results)}: {sample_name}\n" + 
                     f"Overall DSC: {dsc:.4f} | Tissue: {tissue_dsc:.4f} | OS: {os_dsc:.4f} | Vaginal: {vaginal_dsc:.4f}",
                     fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        # Save individual PNG
        png_path = Config.OUTPUT_DIR / f'sample_{idx+1:02d}_{sample_name}.png'
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        
        # Add to PDF
        pdf.savefig(fig, dpi=300, bbox_inches='tight')
        plt.close()

print(f"\n✓ Generated {len(test_results)} visualizations")
print(f"  Individual PNGs saved to: {Config.OUTPUT_DIR}")
print(f"  Combined PDF saved to: {pdf_path}")
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\nFiles generated:")
print(f"  - per_class_performance.csv")
print(f"  - all_test_samples.pdf ({len(test_results)} pages)")
print(f"  - {len(test_results)} individual PNG files")
print("=" * 80)


Generating visualizations for all test samples...


Creating visualizations: 100%|██████████| 12/12 [00:46<00:00,  3.91s/it]



✓ Generated 12 visualizations
  Individual PNGs saved to: ..\..\results\analysis_results
  Combined PDF saved to: ..\..\results\analysis_results\all_test_samples.pdf

ANALYSIS COMPLETE!

Files generated:
  - per_class_performance.csv
  - all_test_samples.pdf (12 pages)
  - 12 individual PNG files
